In [39]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.0 MB/s eta 0:00:00


In [42]:
from groq import Groq

client = Groq(api_key="your-api-key-here")  # Add your Groq API key here

def generate_email(intent, facts, tone):
    prompt = f"""You are a professional email writing assistant.
Study these examples first:

EXAMPLE 1:
Intent: Follow up after job interview
Facts: Interview was on Monday, Position is Software Engineer, Interviewer name is Mr. Sharma
Tone: Formal
Output:
Subject: Follow-Up – Software Engineer Interview

Dear Mr. Sharma,

Thank you for taking the time to meet with me on Monday to discuss the Software Engineer position. I truly enjoyed our conversation and learning more about the role.

Warm regards,
[Your Name]

EXAMPLE 2:
Intent: Request deadline extension
Facts: Project deadline is Friday, Need 3 more days, Reason is additional testing required
Tone: Polite and urgent
Output:
Subject: Request for Deadline Extension – Project Submission

Hi,

I am writing to kindly request a 3-day extension for my project, currently due this Friday. The additional time is needed for thorough testing.

Best regards,
[Your Name]

---

Now generate a professional email for:
Intent: {intent}
Facts: {facts}
Tone: {tone}

Output ONLY the email with Subject line. Nothing else."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print("✅ Groq setup complete!")

✅ Groq setup complete!


In [43]:
intent = "Follow up after business meeting"
facts = "Meeting was yesterday, Discussed Q3 budget, Next meeting on Friday"
tone = "Professional and friendly"

email = generate_email(intent, facts, tone)
print(email)

Subject: Follow-Up – Q3 Budget Meeting Review


In [44]:
import pandas as pd
import time

# 10 Test Scenarios
scenarios = [
    {"intent": "Follow up after job interview", "facts": "Interview on Monday, Position Software Engineer, Interviewer Mr. Sharma", "tone": "Formal"},
    {"intent": "Request deadline extension", "facts": "Deadline Friday, Need 3 more days, Reason additional testing", "tone": "Polite"},
    {"intent": "Thank you after business meeting", "facts": "Meeting was productive, Discussed partnership, Follow up next week", "tone": "Warm and professional"},
    {"intent": "Complaint about delayed delivery", "facts": "Order placed 2 weeks ago, Still not received, Order ID 12345", "tone": "Firm but polite"},
    {"intent": "Request for project proposal", "facts": "Company name TechCorp, Budget 50000, Deadline end of month", "tone": "Formal"},
    {"intent": "Apology for missing meeting", "facts": "Meeting was yesterday, Emergency came up, Want to reschedule", "tone": "Apologetic"},
    {"intent": "Salary increment request", "facts": "Working for 2 years, Performance excellent, Last increment 18 months ago", "tone": "Confident and respectful"},
    {"intent": "Welcome new team member", "facts": "Name is Priya, Joining as Designer, Starting Monday", "tone": "Friendly and warm"},
    {"intent": "Request for feedback on project", "facts": "Project submitted last week, Client name ABC Corp, Need feedback by Friday", "tone": "Professional"},
    {"intent": "Invitation to company event", "facts": "Event on Saturday, Annual dinner, Venue Hotel Taj", "tone": "Formal and exciting"}
]

# Human Reference Emails (ideal outputs)
references = [
    "Subject: Follow-Up – Software Engineer Interview\n\nDear Mr. Sharma,\n\nThank you for meeting me on Monday regarding the Software Engineer position. I am very enthusiastic about this opportunity.\n\nBest regards,\n[Name]",
    "Subject: Request for Deadline Extension\n\nDear Team,\n\nI kindly request a 3-day extension for the project due this Friday, as additional testing is required.\n\nBest regards,\n[Name]",
    "Subject: Thank You – Partnership Discussion\n\nDear Team,\n\nThank you for the productive meeting. I look forward to our follow-up next week regarding the partnership.\n\nWarm regards,\n[Name]",
    "Subject: Complaint – Delayed Delivery (Order #12345)\n\nDear Support,\n\nI placed an order 2 weeks ago (Order ID: 12345) and have not received it yet. Please resolve this urgently.\n\nRegards,\n[Name]",
    "Subject: Project Proposal Request – TechCorp\n\nDear Team,\n\nWe at TechCorp request a project proposal with a budget of 50,000 by end of month.\n\nRegards,\n[Name]",
    "Subject: Apology for Missing Yesterday's Meeting\n\nDear Team,\n\nI sincerely apologize for missing yesterday's meeting due to an emergency. Can we reschedule?\n\nBest regards,\n[Name]",
    "Subject: Request for Salary Increment\n\nDear Manager,\n\nHaving worked for 2 years with excellent performance and no increment in 18 months, I respectfully request a salary review.\n\nRegards,\n[Name]",
    "Subject: Welcome to the Team, Priya!\n\nDear Priya,\n\nWe are thrilled to welcome you as our new Designer starting Monday!\n\nWarm regards,\n[Name]",
    "Subject: Feedback Request – Project Submission\n\nDear ABC Corp,\n\nI submitted the project last week and kindly request your feedback by Friday.\n\nBest regards,\n[Name]",
    "Subject: Invitation – Annual Company Dinner\n\nDear All,\n\nYou are cordially invited to our Annual Dinner on Saturday at Hotel Taj.\n\nRegards,\n[Name]"
]

print("✅ Scenarios ready!")

✅ Scenarios ready!


In [45]:
def metric1_fact_recall(generated, facts):
    """Metric 1: Kitne facts email mein aaye (0-1)"""
    facts_list = [f.strip().lower() for f in facts.split(",")]
    generated_lower = generated.lower()
    found = sum(1 for fact in facts_list if any(word in generated_lower
                for word in fact.split() if len(word) > 3))
    return round(found / len(facts_list), 2)

def metric2_tone_score(generated, tone, client):
    """Metric 2: LLM judge kare tone kitni sahi hai (0-10)"""
    prompt = f"""Rate this email's tone match on a scale of 0-10.
Expected tone: {tone}
Email: {generated}
Reply with ONLY a number between 0-10. Nothing else."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        score = float(response.choices[0].message.content.strip())
        return round(min(max(score, 0), 10) / 10, 2)
    except:
        return 0.5

def metric3_has_subject_and_greeting(generated):
    """Metric 3: Email mein Subject line aur greeting hai ya nahi (0-1)"""
    generated_lower = generated.lower()
    has_subject = "subject:" in generated_lower
    has_greeting = any(word in generated_lower for word in ["dear", "hi", "hello", "greetings"])
    has_closing = any(word in generated_lower for word in ["regards", "sincerely", "best", "thank"])
    score = (int(has_subject) + int(has_greeting) + int(has_closing)) / 3
    return round(score, 2)

print("✅ Metrics ready!")

✅ Metrics ready!


In [46]:
results = []

for i, scenario in enumerate(scenarios):
    print(f"Running scenario {i+1}/10...")

    generated = generate_email(scenario["intent"], scenario["facts"], scenario["tone"])
    time.sleep(2)  # Rate limit se bachne ke liye

    m1 = metric1_fact_recall(generated, scenario["facts"])
    m2 = metric2_tone_score(generated, scenario["tone"], client)
    m3 = metric3_has_subject_and_greeting(generated)
    time.sleep(1)

    results.append({
        "Scenario": i+1,
        "Intent": scenario["intent"],
        "Tone": scenario["tone"],
        "Generated_Email": generated[:200],  # First 200 chars
        "Metric1_Fact_Recall": m1,
        "Metric2_Tone_Score": m2,
        "Metric3_Format_Score": m3,
        "Average_Score": round((m1 + m2 + m3) / 3, 2)
    })
    print(f"  ✅ M1:{m1} | M2:{m2} | M3:{m3}")

df = pd.DataFrame(results)
print("\n📊 RESULTS:")
print(df[["Scenario", "Metric1_Fact_Recall", "Metric2_Tone_Score", "Metric3_Format_Score", "Average_Score"]])
print(f"\n🏆 Overall Average: {df['Average_Score'].mean():.2f}")

# CSV save karo
df.to_csv("evaluation_results.csv", index=False)
print("\n✅ CSV saved: evaluation_results.csv")

Running scenario 1/10...
  ✅ M1:0.67 | M2:0.8 | M3:0.33
Running scenario 2/10...
  ✅ M1:0.33 | M2:0.9 | M3:0.33
Running scenario 3/10...
  ✅ M1:0.33 | M2:0.9 | M3:0.67
Running scenario 4/10...
  ✅ M1:0.67 | M2:0.8 | M3:0.33
Running scenario 5/10...
  ✅ M1:0.33 | M2:0.8 | M3:0.33
Running scenario 6/10...
  ✅ M1:0.33 | M2:0.9 | M3:0.33
Running scenario 7/10...
  ✅ M1:0.33 | M2:0.8 | M3:0.33
Running scenario 8/10...
  ✅ M1:0.33 | M2:0.8 | M3:0.33
Running scenario 9/10...
  ✅ M1:1.0 | M2:0.9 | M3:0.33
Running scenario 10/10...
  ✅ M1:0.67 | M2:0.8 | M3:0.33

📊 RESULTS:
   Scenario  Metric1_Fact_Recall  Metric2_Tone_Score  Metric3_Format_Score  \
0         1                 0.67                 0.8                  0.33   
1         2                 0.33                 0.9                  0.33   
2         3                 0.33                 0.9                  0.67   
3         4                 0.67                 0.8                  0.33   
4         5                 0.33      

In [47]:
# Model B - Zero Shot (bina examples ke)
def generate_email_modelB(intent, facts, tone):
    prompt = f"""Write a professional email.
Intent: {intent}
Facts: {facts}
Tone: {tone}
Output ONLY the email with Subject line."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Model B ke liye 10 scenarios run karo
results_B = []

for i, scenario in enumerate(scenarios):
    print(f"Model B - Scenario {i+1}/10...")

    generated = generate_email_modelB(scenario["intent"], scenario["facts"], scenario["tone"])
    time.sleep(2)

    m1 = metric1_fact_recall(generated, scenario["facts"])
    m2 = metric2_tone_score(generated, scenario["tone"], client)
    m3 = metric3_has_subject_and_greeting(generated)
    time.sleep(1)

    results_B.append({
        "Scenario": i+1,
        "Model": "Model_B_ZeroShot",
        "Metric1_Fact_Recall": m1,
        "Metric2_Tone_Score": m2,
        "Metric3_Format_Score": m3,
        "Average_Score": round((m1 + m2 + m3) / 3, 2)
    })
    print(f"  ✅ M1:{m1} | M2:{m2} | M3:{m3}")

df_B = pd.DataFrame(results_B)

# Comparison
print("\n📊 MODEL A (Few-Shot) Average:", df['Average_Score'].mean().round(2))
print("📊 MODEL B (Zero-Shot) Average:", df_B['Average_Score'].mean().round(2))

# Combined CSV save karo
df['Model'] = 'Model_A_FewShot'
df_combined = pd.concat([df, df_B], ignore_index=True)
df_combined.to_csv("model_comparison.csv", index=False)
print("\n✅ Comparison CSV saved!")

Model B - Scenario 1/10...
  ✅ M1:1.0 | M2:0.9 | M3:1.0
Model B - Scenario 2/10...
  ✅ M1:0.67 | M2:0.8 | M3:0.33
Model B - Scenario 3/10...
  ✅ M1:1.0 | M2:0.9 | M3:1.0
Model B - Scenario 4/10...
  ✅ M1:0.67 | M2:0.8 | M3:0.33
Model B - Scenario 5/10...
  ✅ M1:0.33 | M2:0.9 | M3:0.33
Model B - Scenario 6/10...
  ✅ M1:0.67 | M2:0.9 | M3:0.33
Model B - Scenario 7/10...
  ✅ M1:0.33 | M2:0.8 | M3:0.33
Model B - Scenario 8/10...
  ✅ M1:0.33 | M2:0.9 | M3:0.33
Model B - Scenario 9/10...
  ✅ M1:0.67 | M2:0.9 | M3:0.33
Model B - Scenario 10/10...
  ✅ M1:0.67 | M2:0.9 | M3:0.33

📊 MODEL A (Few-Shot) Average: 0.57
📊 MODEL B (Zero-Shot) Average: 0.66

✅ Comparison CSV saved!
